# Explore: Ollama API

Interactive playground for the Ollama HTTP API itself (`/api/tags`, `/api/show`, `/api/generate`,
`/api/chat`, `/api/ps`) — separate from `explore.ipynb`, which is about this project's RAG pipeline
(embeddings + pgvector retrieval). Nothing here touches pgvector or the `documents` table.

Prerequisites: `ollama serve` running locally, and at least the models in `config.py`
(`OLLAMA_MODEL`, `EMBEDDING_MODEL`) pulled — see the README for `ollama pull` commands.

Kernel: **RAG pgvector (local)** (registered from this project's `.venv` — select it via
Kernel → Change Kernel if it isn't already active).

In [1]:
import json
import time
import urllib.error
import urllib.request

import config

OLLAMA_BASE = config.OLLAMA_URL.rsplit("/api/", 1)[0]


def ollama_get(path: str) -> dict:
    with urllib.request.urlopen(f"{OLLAMA_BASE}{path}", timeout=30) as response:
        return json.loads(response.read().decode("utf-8"))


def ollama_post(path: str, payload: dict, timeout: int = 120) -> dict:
    request = urllib.request.Request(
        f"{OLLAMA_BASE}{path}",
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(request, timeout=timeout) as response:
        return json.loads(response.read().decode("utf-8"))


try:
    ollama_get("/api/tags")
    print(f"Ollama reachable at {OLLAMA_BASE}")
except urllib.error.URLError as exc:
    raise RuntimeError(f"Could not reach Ollama at {OLLAMA_BASE}. Is `ollama serve` running?") from exc

print(f"Generation model: {config.OLLAMA_MODEL}")
print(f"Embedding model:  {config.EMBEDDING_MODEL}")

Ollama reachable at http://127.0.0.1:11434
Generation model: llama3.1:8b
Embedding model:  qwen3-embedding:0.6b


## 1. List local models (`/api/tags`)

Every model `ollama pull`ed on this machine, with size and modified time.

In [2]:
import pandas as pd

models = ollama_get("/api/tags")["models"]
pd.DataFrame(
    [
        {
            "name": m["name"],
            "size_gb": round(m["size"] / 1e9, 2),
            "family": m.get("details", {}).get("family"),
            "parameter_size": m.get("details", {}).get("parameter_size"),
            "quantization": m.get("details", {}).get("quantization_level"),
            "modified_at": m["modified_at"],
        }
        for m in models
    ]
)

,name,size_gb,family,parameter_size,quantization,modified_at
0,qwen3-embedding:4b,2.50,qwen3,4.0B,Q4_K_M,2026-07-18T22:15:53.523358484+05:30
1,qwen3-embedding:0.6b,0.64,qwen3,595.78M,Q8_0,2026-07-18T22:08:15.977243248+05:30
2,qwen3.5:latest,6.59,qwen35,9.7B,Q4_K_M,2026-07-04T06:33:14.139810108+05:30
3,llama3.1:8b,4.92,llama,8.0B,Q4_K_M,2026-03-15T20:47:34.109745207+05:30


## 2. Model details (`/api/show`)

Modelfile, template, and default parameters baked into a given model — useful for seeing what
system prompt or stop tokens a model ships with by default.

In [3]:
show = ollama_post("/api/show", {"model": config.OLLAMA_MODEL})
print("--- parameters ---")
print(show.get("parameters", "(none)"))
print("\n--- template ---")
print(show.get("template", "(none)"))
print("\n--- modelinfo (subset) ---")
for key in ("general.architecture", "general.parameter_count", "llama.context_length"):
    if key in show.get("model_info", {}):
        print(f"{key}: {show['model_info'][key]}")

--- parameters ---
stop                           "<|start_header_id|>"
stop                           "<|end_header_id|>"
stop                           "<|eot_id|>"

--- template ---
{{- if or .System .Tools }}<|start_header_id|>system<|end_header_id|>
{{- if .System }}

{{ .System }}
{{- end }}
{{- if .Tools }}

Cutting Knowledge Date: December 2023

When you receive a tool call response, use the output to format an answer to the orginal user question.

You are a helpful assistant with tool calling capabilities.
{{- end }}<|eot_id|>
{{- end }}
{{- range $i, $_ := .Messages }}
{{- $last := eq (len (slice $.Messages $i)) 1 }}
{{- if eq .Role "user" }}<|start_header_id|>user<|end_header_id|>
{{- if and $.Tools $last }}

Given the following functions, please respond with a JSON for a function call with its proper arguments that best answers the given prompt.

Respond in the format {"name": function name, "parameters": dictionary of argument name and its value}. Do not use variables.

{{

## 3. Generate — non-streaming (`/api/generate`)

Same endpoint `rag.py` uses. `stream=False` blocks until the full response is ready; the reply
includes timing/token-count fields useful for judging latency.

In [5]:
prompt = "Explain what an HNSW index is in two sentences."

t0 = time.time()
result = ollama_post("/api/generate", {"model": config.OLLAMA_MODEL, "prompt": prompt, "stream": False})
elapsed = time.time() - t0

print(result["response"])
print()
eval_count = result.get("eval_count", 0)
eval_duration_s = result.get("eval_duration", 0) / 1e9
tokens_per_sec = eval_count / eval_duration_s if eval_duration_s else float("nan")
print(f"wall time: {elapsed:.2f}s | eval_count: {eval_count} tokens | {tokens_per_sec:.1f} tok/s")

A High-Dimensional Similarity Search Index (HNSW) is a type of data structure used for efficient nearest neighbor search in high-dimensional spaces. It's an indexing technique that uses a graph-like structure to organize the data points, allowing for fast and accurate retrieval of the most similar items to a query point.

wall time: 4.60s | eval_count: 64 tokens | 46.9 tok/s


## 4. Generate — streaming

`stream=True` returns newline-delimited JSON, one object per token (or small group of tokens).
This is what a chat UI would consume to print output as it's generated.

In [6]:
request = urllib.request.Request(
    f"{OLLAMA_BASE}/api/generate",
    data=json.dumps({"model": config.OLLAMA_MODEL, "prompt": prompt, "stream": True}).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST",
)
with urllib.request.urlopen(request, timeout=120) as response:
    for line in response:
        chunk = json.loads(line)
        print(chunk["response"], end="", flush=True)
        if chunk.get("done"):
            print(f"\n\n[done_reason={chunk.get('done_reason')}]")

An HNSW (Hierarchical Navigable Small World) graph index is a data structure that enables efficient nearest neighbor search and k-nearest neighbors search in high-dimensional spaces, achieving a good trade-off between query time and space complexity. The index is constructed by building a hierarchical graph of nodes, where each node represents a data point or a cluster of points, allowing for fast navigation and retrieval of similar items.

[done_reason=stop]


## 5. Sampling options

`temperature`, `top_p`, `top_k`, `num_predict`, and `seed` all go under an `options` dict.
Compare the same prompt at a few temperatures — `seed` is fixed so the only variable is temperature.

In [7]:
creative_prompt = "Write a one-sentence tagline for a local-first vector database."

for temperature in (0.0, 0.7, 1.4):
    result = ollama_post(
        "/api/generate",
        {
            "model": config.OLLAMA_MODEL,
            "prompt": creative_prompt,
            "stream": False,
            "options": {"temperature": temperature, "seed": 42, "num_predict": 50},
        },
    )
    print(f"temperature={temperature}: {result['response'].strip()}")

temperature=0.0: "Store your data with precision, not latency: local-first vector databases for the modern application."
temperature=0.7: "Store, query, and serve your data with the speed of vectors, where every record is a neighbor."
temperature=1.4: "Store and query your data, close to home: fast, reliable storage for what matters most, locally."


## 6. Chat endpoint (`/api/chat`)

`/api/chat` takes a `messages` list (`system`/`user`/`assistant` roles) instead of a single prompt
string, and manages the conversation turn structure for you — closer to what a multi-turn assistant
needs than raw `/api/generate`.

In [8]:
messages = [
    {"role": "system", "content": "You are a terse assistant. Answer in one sentence."},
    {"role": "user", "content": "What is pgvector?"},
]
result = ollama_post("/api/chat", {"model": config.OLLAMA_MODEL, "messages": messages, "stream": False})
reply = result["message"]["content"]
print(f"assistant: {reply}")

# Append the reply and ask a follow-up in the same conversation.
messages.append({"role": "assistant", "content": reply})
messages.append({"role": "user", "content": "Does it support more than one distance metric?"})
result = ollama_post("/api/chat", {"model": config.OLLAMA_MODEL, "messages": messages, "stream": False})
print(f"assistant: {result['message']['content']}")

assistant: pgvector is an open-source vector database for PostgreSQL that allows for the storage and querying of high-dimensional vectors, such as those used in natural language processing and computer vision applications.
assistant: Yes, pgvector supports multiple distance metrics, including dot product, Euclidean, Manhattan, Jaccard, Cosine, and Haversine.


## 7. Structured output (`format="json"`)

Ask the model to return valid JSON matching a shape you describe in the prompt, then parse it.

In [9]:
structured_prompt = (
    'Return a JSON object with keys "term" and "definition" defining "cosine similarity". '
    "Respond with only the JSON object."
)
result = ollama_post(
    "/api/generate",
    {"model": config.OLLAMA_MODEL, "prompt": structured_prompt, "stream": False, "format": "json"},
)
parsed = json.loads(result["response"])
parsed

{'term': 'Cosine Similarity',
 'definition': 'A measure of similarity between two vectors by taking the cosine of the angle between them, often used in natural language processing to compare the meaning of text.'}

## 8. Currently loaded models (`/api/ps`)

Which models Ollama has resident in memory right now, and how much VRAM/RAM each is using — handy
after running the cells above to see what's actually loaded.

In [10]:
ps = ollama_get("/api/ps")["models"]
pd.DataFrame(
    [
        {
            "name": m["name"],
            "size_vram_gb": round(m.get("size_vram", 0) / 1e9, 2),
            "expires_at": m["expires_at"],
        }
        for m in ps
    ]
) if ps else print("No models currently loaded.")

,name,size_vram_gb,expires_at
0,llama3.1:8b,5.46,2026-07-19T08:53:56.70502+05:30
